# Step 8 — Pre-Mortem Generator
**Tough Talks · Phase 3**

Goal: prove Gemma 4 E2B (text-only) can generate three SPECIFIC failure scenarios for an upcoming difficult conversation, each parameterised so they can later feed into Step 7's adversarial persona simulator.

The output is one structured JSON object matching `data/schemas/premortem.schema.json`:

- `goal` — paraphrased user goal.
- `failure_scenarios[3]` — exactly three numbered scenarios, each with:
  - `title`, `description`, `likely_trigger`, `destabilization_risk`
  - `simulation_parameters.{resistance_type, escalation_ceiling, opening_move}` — same `resistance_type` enum as `persona_reply.schema.json`, so a scenario can be lifted straight into `PersonaSimConfig` to drive a Step-7 practice round.

**Architecture** — same hybrid-runtime pattern as Steps 04 / 05 / 06 / 07:
- Runtime in `backend/core/_runtime/premortem.py`. Notebook is a thin driver.
- Single prompt-based JSON call (analytical, not multi-turn).
- Two-shot retry: greedy first, then a single light-sampling pass (`temperature=0.3, top_p=0.9, top_k=64`) on JSON / validation failure.
- System-contract enforcement in code: `scenario_id` forced to 1/2/3 by index, `resistance_type` synonym-normalised against the persona-sim canonical enum (`denial`→`deny`, `counterattack`→`counter_attack`, `silence`→`silent`, etc.), `destabilization_risk` and `escalation_ceiling` clamped to `[0, 1]`, required string fields rejected when empty.
- `enable_thinking` is exposed as a knob, defaulting to `False`. The final cell A/B-compares thinking-on vs thinking-off — pre-mortem is the first analytical / reflection-family task we hit, and the open hypothesis in `knowledge/phases/hypotheses.md` says E2B *might* benefit from thinking on these tasks (the `[[hypothesis-persona-thinking-helps]]` refutation was specifically on persona simulation where in-character voice gets washed out — analytical tasks have no character to wash out).

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `generate_premortem()` returns a schema-conforming dict with exactly 3 scenarios.
3. Each scenario's `resistance_type` lands inside the closed enum (synonym normaliser silently corrects model drift).
4. `destabilization_risk` and `escalation_ceiling` are valid numbers in `[0, 1]`.
5. Each scenario has a non-empty `title`, `description`, `likely_trigger`, and `opening_move`.
6. `simulation_parameters` round-trips into `PersonaSimConfig` cleanly (final cell demonstrates the wiring without paying a second model call).
7. The A/B cell renders both `enable_thinking=False` and `enable_thinking=True` outputs side-by-side

**Note on inputs.** Both the PersonVault profile for Jamie and a representative TalkDNA profile for the user are hardcoded inline (matching Step 06's v2 output and Step 05's v2 output respectively) so this notebook doesn't re-pay Steps 05 / 06's combined ~25-minute LLM cost on every run. In production these come from `analyze_person_vault()` and `analyze_talk_dna()`.

In [1]:
# ── 0. Install / upgrade dependencies ───────────────────────────────
# Text-only path — no audio libs required. Same rule as Steps 05 / 06 /
# 07: bump only transformers + accelerate on Colab / Kaggle (bumping
# torch breaks the pre-installed torchvision / CUDA pairing). After this
# first run, RESTART THE KERNEL before continuing if you actually
# upgraded transformers — the already-imported version won't pick up
# the change.

!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 120.9 MB/s eta 0:00:0000:010:01


In [20]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────
# Same shim as Steps 01–07 — auto-clones / refreshes on Colab / Kaggle
# and clears any cached `backend.*` modules so the imports below pick
# up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Refreshing /content/tough_talks from origin
Repo root: /content/tough_talks


In [21]:
# ── 2. Imports ───────────────────────────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_RESISTANCE_TYPES,
    DEFAULT_MODEL_ID,
    LoadConfig,
    PersonaSimConfig,
    PremortemConfig,
    PremortemError,
    REQUIRED_SCENARIO_COUNT,
    format_persona_profile_block,
    format_talk_dna_block,
    generate_premortem,
    load_model,
)

In [22]:
# ── 3. Configuration ─────────────────────────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "premortem.schema.json"

# PersonVault profile for Jamie. Same v2 dict as Step 07 — defensive
# in Conv A, softened to assertive in B, accumulated triggers /
# de-escalation keys / deflections across both conversations. Inlined
# so this notebook doesn't re-pay Step 06's ~15-minute LLM cost on
# every run. In production this comes from `analyze_person_vault()`.
JAMIE_PROFILE = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "suggesting escalation when blockers are still open",
            "implying missed communication is one-sided",
            "setting hard deadlines without acknowledging blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
            "acknowledging the tightness of the timeline",
            "proposing a concrete next-step the user will own",
        ],
        "common_deflections": [
            "I told you",
            "Don't blame me",
            "Don't put this on me",
            "You weren't there",
            "staging tables aren't done",
        ],
        "responds_best_to": (
            "Clear, actionable next steps tied to specific milestones. "
            "Responds well when the user accepts responsibility for "
            "communication gaps."
        ),
    },
}

# Representative TalkDNA profile for the user — matches the v2 shape
# Step 05 produces (apology-prone, hedges, silence_under_pressure
# true). Inlined for the same reason as Jamie's profile: avoid re-
# paying Step 05's cost on every notebook run. In production this
# comes from `analyze_talk_dna()`.
USER_TALK_DNA = {
    "user_id": "local",
    "version": 2,
    "conversation_count": 2,
    "patterns": {
        "filler_phrases": ["I just feel like", "kind of", "sort of"],
        "apology_rate": 0.57,
        "silence_under_pressure": True,
        "sarcasm_frequency": "low",
        "escalation_triggers": [
            "feeling dismissed",
            "being told to wait without a reason",
        ],
        "avg_turn_length_words": 15.7,
    },
    "strengths": [
        "clear_problem_statement",
        "acknowledges_mistakes",
    ],
    "weaknesses": [
        "over_apologizes",
        "hedges_before_vulnerable_statements",
    ],
    "updated_at": "2026-05-14T00:00:00Z",
}

# What the user wants out of the upcoming conversation.
USER_GOAL = (
    "Get Jamie to commit to delivering the staging-tables data by "
    "Wednesday EOD and to acknowledge that the escalation last week "
    "needed to land more clearly — not just be sent."
)

# What the user is actually preparing to discuss. The pre-mortem
# generator reasons over THIS context plus Jamie's profile plus the
# user's own TalkDNA to produce three failure scenarios.
CONVERSATION_DESCRIPTION = (
    "Tomorrow I'm meeting 1:1 with Jamie (a colleague on the data "
    "team) to discuss what happened with last week's report. The "
    "report missed its Tuesday deadline because the staging tables "
    "weren't ready. Jamie says they emailed me a heads-up that I "
    "didn't see in time. I want to use this meeting to (a) agree on "
    "a clear escalation channel for blocker visibility going forward, "
    "and (b) commit to a concrete delivery date for the staging "
    "tables this sprint. Jamie tends to get defensive when I bring up "
    "missed deadlines and frequently deflects to past communication "
    "they say I missed. I know I tend to over-apologise when "
    "challenged, which Jamie can read as 'OK, you agree with me'."
)

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Persona            : {JAMIE_PROFILE['name']} ({JAMIE_PROFILE['relationship_type']})")
print(f"Resistance enum    : {ALLOWED_RESISTANCE_TYPES}")
print(f"Scenarios required : {REQUIRED_SCENARIO_COUNT}")

Model              : google/gemma-4-E2B-it
Device             : cuda
Persona            : Jamie (colleague)
Resistance enum    : ('deflect', 'guilt_trip', 'deny', 'counter_attack', 'silent', 'concede')
Scenarios required : 3


In [23]:
# ── 4. Preview the rendered context blocks (no model needed) ──────────────
# Both `format_persona_profile_block` and `format_talk_dna_block` are
# what the runtime feeds into the prompt. Rendering them here is purely
# diagnostic — it lets us verify both profiles are well-formed before
# paying for the model load.

print("=" * 76)
print("PERSON PROFILE BLOCK (Jamie)")
print("=" * 76)
print(format_persona_profile_block(JAMIE_PROFILE))

print()
print("=" * 76)
print("TALK DNA BLOCK (user)")
print("=" * 76)
print(format_talk_dna_block(USER_TALK_DNA))

PERSON PROFILE BLOCK (Jamie)
- communication_style: defensive
- emotional_triggers (USER-side cues that escalate / make you defensive): ['citing past commitments', 'suggesting escalation when blockers are still open', 'implying missed communication is one-sided', 'setting hard deadlines without acknowledging blockers']
- de_escalation_keys (USER-side moves that calm / open you up): ['explicitly disowning blame', 'reframing as joint problem-solving', 'acknowledging the tightness of the timeline', 'proposing a concrete next-step the user will own']
- common_deflections (YOUR habitual evasion phrases): ['I told you', "Don't blame me", "Don't put this on me", "You weren't there", "staging tables aren't done"]
- responds_best_to: Clear, actionable next steps tied to specific milestones. Responds well when the user accepts responsibility for communication gaps.

TALK DNA BLOCK (user)
- weaknesses (USER habits the opponent could exploit): ['over_apologizes', 'hedges_before_vulnerable_statemen

In [6]:
# ── 5. Load the text-only processor + model ─────────────────────────
# Pre-mortem reasons over WORDS — same rationale as TalkDNA /
# PersonVault / persona-sim. The `multimodal=False` (default) path
# loads `AutoModelForCausalLM`, which is lighter on VRAM and slightly
# faster than the multimodal class. Live Mode (Phase 5+) will reuse
# this same loaded model across components, so the load cost is
# amortised.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [24]:
# ── 6. Generate the pre-mortem (greedy, thinking=False) ───────────────────
# Single prompt-based JSON call — no rolling history, no per-turn loop
# (unlike Step 7's persona simulator). The runtime threads the
# conversation context, user goal, person profile, and TalkDNA into the
# `data/prompts/premortem.md` template, calls `chat()` greedy, parses
# the JSON, validates / coerces against the schema, and returns a
# schema-conforming dict.

cfg = PremortemConfig(
    conversation_description=CONVERSATION_DESCRIPTION,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=False,
)

try:
    premortem = generate_premortem(processor, model, cfg=cfg)
except PremortemError as exc:
    print(f"FAILED: {exc}")
    for attempt in getattr(exc, "attempts", []):
        print(f"  attempt ({attempt['sampling']}): {attempt['error']}")
        print(f"  raw_head: {attempt['raw_head']!r}")
    raise

# Render the result inline so the structure is visible turn-by-turn,
# not just in the validation table below.
print(f"GOAL: {premortem['goal']}\n")
for s in premortem["failure_scenarios"]:
    sim = s["simulation_parameters"]
    print(f"--- Scenario {s['scenario_id']}: {s['title']} ---")
    print(f"  description     : {s['description']}")
    print(f"  likely_trigger  : {s['likely_trigger']}")
    print(f"  destabilization : {s['destabilization_risk']:.2f}")
    print(f"  resistance_type : {sim['resistance_type']}")
    print(f"  escalation_ceil : {sim['escalation_ceiling']:.2f}")
    print(f"  opening_move    : {sim['opening_move']}")
    print()

GOAL: To secure a concrete delivery date for the staging-tables data by Wednesday EOD and ensure Jamie acknowledges the need for clearer escalation on past issues.

--- Scenario 1: Blame Shifting and Deflection ---
  description     : When the user clearly states the need for a commitment, Jamie immediately pivots the focus back to the past communication failure, using their common deflection to avoid the current commitment. They refuse to engage with the future action item.
  likely_trigger  : User explicitly states the need for a concrete delivery date.
  destabilization : 0.60
  resistance_type : deflect
  escalation_ceil : 0.70
  opening_move    : I told you last week that the staging tables weren't ready, and I was waiting on X.

--- Scenario 2: Weaponizing Past Commitments ---
  description     : The user attempts to reframe the issue as a joint problem-solving exercise, but Jamie responds by citing a previous, unrelated commitment or past failure to make the current request seem

In [25]:
# ── 7. Schema validation + results table ──────────────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05 / 06
# / 07). Per scenario:
#   - required schema fields present (scenario_id, title, description,
#     likely_trigger, destabilization_risk, simulation_parameters)
#   - simulation_parameters required keys (resistance_type,
#     escalation_ceiling, opening_move)
#   - resistance_type in the persona-sim canonical enum
#   - destabilization_risk and escalation_ceiling in [0, 1]
#   - title / description / likely_trigger / opening_move are non-empty
#     strings
# Cross-scenario:
#   - exactly REQUIRED_SCENARIO_COUNT scenarios
#   - scenario_id sequence is 1, 2, 3 in order (runtime-forced)
#   - destabilization_risk values vary (not three identical numbers)
#   - resistance_type variety (≥ 2 distinct values across the 3
#     scenarios — the prompt explicitly asks for variety; identical
#     resistance_types across all three is a quality smell, not a
#     hard fail)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
SCENARIO_REQUIRED = schema["properties"]["failure_scenarios"]["items"]["required"]
SIM_REQUIRED      = schema["properties"]["failure_scenarios"]["items"][
    "properties"
]["simulation_parameters"]["required"]
RESISTANCE_ENUM   = schema["properties"]["failure_scenarios"]["items"][
    "properties"
]["simulation_parameters"]["properties"]["resistance_type"]["enum"]


def _validate_scenario(s: dict) -> list[str]:
    errs: list[str] = []
    for k in SCENARIO_REQUIRED:
        if k not in s:
            errs.append(f"missing required key {k!r}")
    risk = s.get("destabilization_risk")
    if isinstance(risk, (int, float)):
        if not (0.0 <= float(risk) <= 1.0):
            errs.append(f"destabilization_risk {risk} outside [0, 1]")
    elif risk is not None:
        errs.append(f"destabilization_risk not numeric: {risk!r}")
    for sk in ("title", "description", "likely_trigger"):
        v = s.get(sk)
        if not isinstance(v, str) or not v.strip():
            errs.append(f"{sk} empty or non-string")
    sim = s.get("simulation_parameters")
    if not isinstance(sim, dict):
        errs.append("simulation_parameters not an object")
        return errs
    for k in SIM_REQUIRED:
        if k not in sim:
            errs.append(f"simulation_parameters missing {k!r}")
    rt = sim.get("resistance_type")
    if rt is not None and rt not in RESISTANCE_ENUM:
        errs.append(f"resistance_type {rt!r} not in {RESISTANCE_ENUM}")
    ceil = sim.get("escalation_ceiling")
    if isinstance(ceil, (int, float)):
        if not (0.0 <= float(ceil) <= 1.0):
            errs.append(f"escalation_ceiling {ceil} outside [0, 1]")
    elif ceil is not None:
        errs.append(f"escalation_ceiling not numeric: {ceil!r}")
    om = sim.get("opening_move")
    if not isinstance(om, str) or not om.strip():
        errs.append("opening_move empty or non-string")
    return errs


scenarios = premortem["failure_scenarios"]
per_scenario_errors = [_validate_scenario(s) for s in scenarios]
n_clean = sum(1 for errs in per_scenario_errors if not errs)

ids        = [s.get("scenario_id") for s in scenarios]
risks      = [s.get("destabilization_risk", 0.0) for s in scenarios]
ceilings   = [s["simulation_parameters"].get("escalation_ceiling", 0.0)
              for s in scenarios]
rts        = [s["simulation_parameters"].get("resistance_type") for s in scenarios]
risk_var   = (max(risks) - min(risks)) if risks else 0.0
rt_variety = len(set(rts))

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",        True,                                                f"{n_params:.1f}B params on {model.device}"),
    ("scenario_count_correct",   len(scenarios) == REQUIRED_SCENARIO_COUNT,           f"{len(scenarios)}/{REQUIRED_SCENARIO_COUNT}"),
    ("scenario_id_sequence",     ids == list(range(1, REQUIRED_SCENARIO_COUNT + 1)),  f"{ids}"),
    ("schema_valid_per_scenario", n_clean == len(scenarios),                           f"{n_clean}/{len(scenarios)} clean"),
    ("resistance_in_enum",       all(rt in RESISTANCE_ENUM for rt in rts),            f"{rts}"),
    ("destabilization_in_bounds", all(0.0 <= float(r) <= 1.0 for r in risks),         f"{[round(r, 2) for r in risks]}"),
    ("escalation_ceiling_in_bounds", all(0.0 <= float(c) <= 1.0 for c in ceilings),   f"{[round(c, 2) for c in ceilings]}"),
    ("destabilization_has_variance", risk_var > 0.05,                                 f"max - min = {risk_var:.2f}"),
    ("resistance_type_variety",  rt_variety >= 2,                                     f"{rt_variety} distinct value(s) across {len(rts)} scenarios"),
    ("goal_present",             bool(premortem.get("goal")),                         premortem.get("goal", "")[:60] + "..."),
    ("premortem_id_attached",    isinstance(premortem.get("premortem_id"), str)
                                  and premortem["premortem_id"].startswith("premortem_"),
                                                                                      f"{premortem.get('premortem_id', '<missing>')}"),
]

print("=" * 76)
print("STEP 8 RESULTS — Gemma 4 Pre-Mortem Generator")
print("=" * 76)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:32s}  {note}")
    if not ok:
        all_ok = False

if any(errs for errs in per_scenario_errors):
    print("\nSchema errors:")
    for i, errs in enumerate(per_scenario_errors, start=1):
        if not errs:
            continue
        print(f"  Scenario {i}:")
        for line in errs:
            print(f"    - {line}")

print()
print("OVERALL:", "READY FOR STEP 9" if all_ok else "FIX FAILURES ABOVE")

STEP 8 RESULTS — Gemma 4 Pre-Mortem Generator
[PASS]  text_model_loaded                 5.1B params on cuda:0
[PASS]  scenario_count_correct            3/3
[PASS]  scenario_id_sequence              [1, 2, 3]
[PASS]  schema_valid_per_scenario         3/3 clean
[PASS]  resistance_in_enum                ['deflect', 'guilt_trip', 'deny']
[PASS]  destabilization_in_bounds         [0.6, 0.85, 0.4]
[PASS]  escalation_ceiling_in_bounds      [0.7, 0.9, 0.5]
[PASS]  destabilization_has_variance      max - min = 0.45
[PASS]  resistance_type_variety           3 distinct value(s) across 3 scenarios
[PASS]  goal_present                      To secure a concrete delivery date for the staging-tables da...
[PASS]  premortem_id_attached             premortem_bae271c5d258

OVERALL: READY FOR STEP 9


In [26]:
# ── 8. A/B: enable_thinking=False vs enable_thinking=True (same context) ──
# Tests the open hypothesis — does Gemma 4's thinking channel help on
# analytical / reflection-family tasks (debrief, premortem, aftermath)?
# Pre-mortem has no "character" to wash out, so this is the first place
# the hypothesis is genuinely live again after Step 7's persona-sim
# refutation.
#
# Run 1 finding: `max_new_tokens=1024` was tight enough that the
# thinking branch failed JSON parsing on both greedy and light-sampling
# attempts (truncated mid-object after the thought trace ate most of the
# budget). Premortem's JSON body alone is ~800 tokens. Bumped to 2048
# for the thinking branch to give a verbose thought trace AND the full
# closing brace room.
#
# Also wrapping the thinking call in a try/except so a future failure
# prints the raw model output (which the runtime attaches to
# `exc.attempts`) without needing a re-run to see what happened.

cfg_no_thinking = PremortemConfig(
    conversation_description=CONVERSATION_DESCRIPTION,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=False,
)
cfg_thinking = PremortemConfig(
    conversation_description=CONVERSATION_DESCRIPTION,
    user_goal=USER_GOAL,
    person_profile=JAMIE_PROFILE,
    talk_dna_profile=USER_TALK_DNA,
    enable_thinking=True,
    max_new_tokens=2048,
)

pre_no_thinking = generate_premortem(processor, model, cfg=cfg_no_thinking)

pre_thinking = None
thinking_err = None
try:
    pre_thinking = generate_premortem(processor, model, cfg=cfg_thinking)
except PremortemError as exc:
    thinking_err = exc

print("=" * 76)
print("A/B — enable_thinking on the same conversation context")
print("=" * 76)

print("\n--- thinking=False (default) ---")
print(f"  goal: {pre_no_thinking['goal']}")
for s in pre_no_thinking["failure_scenarios"]:
    sim = s["simulation_parameters"]
    print(
        f"  S{s['scenario_id']} [{sim['resistance_type']:<14s} "
        f"risk={s['destabilization_risk']:.2f} "
        f"ceiling={sim['escalation_ceiling']:.2f}] {s['title']}"
    )
    print(f"      opening_move: {sim['opening_move']}")

print("\n--- thinking=True ---")
if pre_thinking is not None:
    print(f"  goal: {pre_thinking['goal']}")
    for s in pre_thinking["failure_scenarios"]:
        sim = s["simulation_parameters"]
        print(
            f"  S{s['scenario_id']} [{sim['resistance_type']:<14s} "
            f"risk={s['destabilization_risk']:.2f} "
            f"ceiling={sim['escalation_ceiling']:.2f}] {s['title']}"
        )
        print(f"      opening_move: {sim['opening_move']}")
else:
    print(f"  FAILED: {thinking_err}")
    for attempt in getattr(thinking_err, "attempts", []):
        print(f"\n  attempt ({attempt['sampling']}): {attempt['error']}")
        print(f"  raw_head (first 500 chars):")
        print(f"  {attempt['raw_head']!r}")

A/B — enable_thinking on the same conversation context

--- thinking=False (default) ---
  goal: To secure a concrete delivery date for the staging-tables data by Wednesday EOD and ensure Jamie acknowledges the need for clearer escalation on past issues.
  S1 [deflect        risk=0.60 ceiling=0.70] Blame Shifting and Deflection
      opening_move: I told you last week that the staging tables weren't ready, and I was waiting on X.
  S2 [guilt_trip     risk=0.85 ceiling=0.90] Weaponizing Past Commitments
      opening_move: We need to agree on a clear escalation channel for blocker visibility going forward.
  S3 [deny           risk=0.40 ceiling=0.50] Over-Apology Exploitation
      opening_move: I'm sorry about last week; I really messed up on the follow-up.

--- thinking=True ---
  goal: To secure a concrete delivery date for the staging-tables data by Wednesday EOD and establish a clear escalation channel for future blockers.
  S1 [deflect        risk=0.40 ceiling=0.50] The Blame Redi

In [ ]:
# ── 9. Pre-mortem → persona-sim wiring preview (no extra model call) ──────
# Demonstrates the schema contract end-to-end: a generated scenario's
# `simulation_parameters` is shaped to be lifted directly into a
# `PersonaSimConfig`, with `opening_move` pre-seeded as the persona's
# first assistant turn. The user can then reply, and Step 7's
# `generate_persona_reply` continues the rolling history from there.
#
# This cell only BUILDS the config + history — it does not call the
# model again. The point is to prove the contract round-trips; the
# practice round itself belongs to Step 7's notebook.
#
# Prefers cell-8's thinking-on premortem when available, otherwise
# falls back to cell-6's thinking-off premortem. The N=2 finding for
# this step is that thinking-on produces noticeably cleaner output on
# pre-mortem (better role discipline in `opening_move`, better verbatim
# `common_deflection` integration) — so when both branches succeeded,
# the demo uses the stronger source. The contract test itself doesn't
# depend on which source we pick.

_source_premortem = (
    pre_thinking
    if "pre_thinking" in globals() and pre_thinking is not None
    else premortem
)
_source_label = (
    "cell-8 thinking=True"
    if "pre_thinking" in globals() and pre_thinking is not None
    else "cell-6 thinking=False"
)

# Pick the scenario with the highest destabilization_risk to rehearse.
scenarios_by_risk = sorted(
    _source_premortem["failure_scenarios"],
    key=lambda s: s["destabilization_risk"],
    reverse=True,
)
worst = scenarios_by_risk[0]
worst_sim = worst["simulation_parameters"]

# Build the persona-sim handoff. Notice we DON'T copy
# `escalation_ceiling` into `PersonaSimConfig` — Step 7's config
# doesn't have that field. The ceiling is a target for the persona to
# build toward; the runtime tracks current `escalation_level` per
# reply. So `escalation_ceiling` is metadata the practice UI / Step 9
# debrief can use to score how close the actual run got to the worst
# case.
persona_cfg = PersonaSimConfig(
    persona_profile=JAMIE_PROFILE,
    user_goal=_source_premortem["goal"],
    enable_thinking=False,
)

# Pre-seed the history with the opening move as the persona's first
# turn. `generate_persona_reply` would then take the user's response
# as its next input and produce reply #2.
seeded_history = [
    {
        "speaker": "persona",
        "persona_name": JAMIE_PROFILE["name"],
        "reply": worst_sim["opening_move"],
        "resistance_type": worst_sim["resistance_type"],
        # Estimate starting escalation as half of the ceiling — the
        # opening move shouldn't already be at the ceiling, the
        # ceiling is where the conversation could land if it goes badly.
        "escalation_level": round(worst_sim["escalation_ceiling"] / 2, 2),
    }
]

print("=" * 76)
print("PRE-MORTEM → PERSONA-SIM HANDOFF (worst-case scenario)")
print("=" * 76)
print(f"Source             : {_source_label}")
print(f"Selected scenario  : #{worst['scenario_id']} — {worst['title']}")
print(f"  destabilization  : {worst['destabilization_risk']:.2f}")
print(f"  resistance_type  : {worst_sim['resistance_type']}")
print(f"  escalation ceiling: {worst_sim['escalation_ceiling']:.2f}")
print()
print("PersonaSimConfig:")
print(f"  persona_profile  : <Jamie v2 PersonVault>")
print(f"  user_goal        : {persona_cfg.user_goal}")
print(f"  enable_thinking  : {persona_cfg.enable_thinking}")
print()
print("Seeded history (persona opens with worst-case line):")
print(json.dumps(seeded_history, indent=2, ensure_ascii=False))
print()
print(
    "Step 9 / live mode would now call "
    "`generate_persona_reply(processor, model, user_reply, "
    "history=seeded_history, cfg=persona_cfg)` "
    "to continue the practice round from this opening."
)